### Model cost/watt as function of three features: time (days), size (kilowatts), state

Add categorical feature sate to model (cost ~ days, size, state(C))

## Add feature *size_kw*  and *state* to model

In [1]:
### %matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 8))
np.set_printoptions(precision=4, suppress=False)
# please show all columns
pd.set_option("display.max_columns", 60)
import seaborn as sns
sns.set()

In [4]:
# Import sklearn stuff

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline,  make_pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

In [5]:
# read cleaned data
dfModelAll = pd.read_csv('../local/data/LBNL_openpv_tts_data/ModelAll.csv', index_col='row', dtype={'state':'category'})

In [6]:
dfModelAll.head()

,num_days,size_kw,state,cost_per_watt,scaleSize
row,,,,,
0,0.0,2.2824,CA,10.734315,0
1,21.0,1.8504,CA,11.108701,0
2,26.0,2.3076,CA,8.667013,0
3,84.0,2.3316,CA,13.270286,0
4,111.0,0.9300,CA,14.654839,0


In [7]:
dfModelAll.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 364212 entries, 0 to 364211
Data columns (total 5 columns):
num_days         364212 non-null float64
size_kw          364212 non-null float64
state            364212 non-null category
cost_per_watt    364212 non-null float64
scaleSize        364212 non-null int64
dtypes: category(1), float64(3), int64(1)
memory usage: 14.2 MB


In [8]:
dfModelAll.state.unique()

[CA, OR, AZ, NY, MN, ..., MD, CT, FL, NM, AR]
Length: 19
Categories (19, object): [CA, OR, AZ, NY, ..., CT, FL, NM, AR]

##### We have a choice to deal with encoding of categorical variable 'state' with pandas or in the pipeline.  I don't want to make polynomial terms for the state.

### make a small set to try out the pipeline

#### let's do the one hot encoding with pandas

In [9]:
dfLittle = dfModelAll.sample(n=1000); dfLittle.head()

,num_days,size_kw,state,cost_per_watt,scaleSize
row,,,,,
282627,6640.0,5.865,CA,4.728048,2
22558,3225.0,4.900,NJ,8.500000,1
50856,4126.0,7.425,CA,7.889918,2
360468,6916.0,7.035,CA,4.718348,2
31636,3566.0,3.240,CA,9.413580,1


In [10]:
little = pd.get_dummies(dfLittle, drop_first=True)

In [11]:
little.head()

,num_days,size_kw,cost_per_watt,scaleSize,state_AZ,state_CA,state_CT,state_DE,state_FL,state_MA,state_MD,state_MN,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OR,state_PA,state_TX,state_VT,state_WI
row,,,,,,,,,,,,,,,,,,,,,,
282627,6640.0,5.865,4.728048,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22558,3225.0,4.900,8.500000,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
50856,4126.0,7.425,7.889918,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
360468,6916.0,7.035,4.718348,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
31636,3566.0,3.240,9.413580,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


#### okay, I'm one-hot now, but how to do the polynomial transform in pipeline without including state dummy vars?

It seems like I want a FeatureUnion to do polynomial expansion on num_days, size_kw, on one side and do nothing to the state_AZ, etc.

In [12]:
# https://wkirgsn.github.io/2018/02/15/pandas-pipelines/

# define a simple transformer
class SimpleTransformer(BaseEstimator, TransformerMixin):
    """Apply given transformation."""
    def __init__(self, trans_func, untrans_func, columns):
        self.transform_func = trans_func
        self.inverse_transform_func = untrans_func
        self.cols = columns

    def fit(self, x, y=None):
        return self

    def transform(self, x):
        x = self._get_selection(x)
        return self.transform_func(x) if callable(self.transform_func) else x

    def inverse_transform(self, x):
        return self.inverse_transform_func(x) \
            if callable(self.inverse_transform_func) else x

    def _get_selection(self, df):
        assert isinstance(df, pd.DataFrame)
        return df[self.cols]

    def get_feature_names(self):
        return self.cols

In [13]:
# this one does nothing; just sends input to output
# Pipeline wants a list of tuples, (name(string), and an instance (of estimator/transformer))
nullpipe = Pipeline([('simp', SimpleTransformer(None, None, little.columns))])
# ha, ha.  It doesn't blow up

In [14]:
nullpipe.fit_transform(little).head()

,num_days,size_kw,cost_per_watt,scaleSize,state_AZ,state_CA,state_CT,state_DE,state_FL,state_MA,state_MD,state_MN,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OR,state_PA,state_TX,state_VT,state_WI
row,,,,,,,,,,,,,,,,,,,,,,
282627,6640.0,5.865,4.728048,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22558,3225.0,4.900,8.500000,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
50856,4126.0,7.425,7.889918,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
360468,6916.0,7.035,4.718348,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
31636,3566.0,3.240,9.413580,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [15]:
little.columns

Index(['num_days', 'size_kw', 'cost_per_watt', 'scaleSize', 'state_AZ',
       'state_CA', 'state_CT', 'state_DE', 'state_FL', 'state_MA', 'state_MD',
       'state_MN', 'state_NH', 'state_NJ', 'state_NM', 'state_NV', 'state_NY',
       'state_OR', 'state_PA', 'state_TX', 'state_VT', 'state_WI'],
      dtype='object')

In [16]:
selNumVarsPipe = Pipeline([('simp', SimpleTransformer(None, None, ['num_days', 'size_kw']))])

In [17]:
selNumVarsPipe.fit_transform(little).head()

,num_days,size_kw
row,,
282627,6640.0,5.865
22558,3225.0,4.900
50856,4126.0,7.425
360468,6916.0,7.035
31636,3566.0,3.240


#### Note: FeatureUnion transformation returns np.ndarray

In [18]:
### make it twice as wide
double = FeatureUnion([('num1', selNumVarsPipe),
                       ('num2', selNumVarsPipe)])
doubleDown = double.fit_transform(little)
doubleDown[:5]

array([[6.640e+03, 5.865e+00, 6.640e+03, 5.865e+00],
       [3.225e+03, 4.900e+00, 3.225e+03, 4.900e+00],
       [4.126e+03, 7.425e+00, 4.126e+03, 7.425e+00],
       [6.916e+03, 7.035e+00, 6.916e+03, 7.035e+00],
       [3.566e+03, 3.240e+00, 3.566e+03, 3.240e+00]])

In [19]:
type(doubleDown)

numpy.ndarray

In [20]:
### invoke right after construction
FeatureUnion([('one', Pipeline([('simp', SimpleTransformer(None, None, ['num_days', 'size_kw']))])),
              ('two', Pipeline([('simp', SimpleTransformer(None, None, ['num_days', 'size_kw']))]))
             ]).fit_transform(little)[:5]

array([[6.640e+03, 5.865e+00, 6.640e+03, 5.865e+00],
       [3.225e+03, 4.900e+00, 3.225e+03, 4.900e+00],
       [4.126e+03, 7.425e+00, 4.126e+03, 7.425e+00],
       [6.916e+03, 7.035e+00, 6.916e+03, 7.035e+00],
       [3.566e+03, 3.240e+00, 3.566e+03, 3.240e+00]])

In [21]:
make_pipeline(SimpleTransformer(None, None, ['num_days', 'size_kw']), 
              StandardScaler()).fit_transform(little)[:5]

array([[ 0.8791, -0.077 ],
       [-1.9219, -0.3761],
       [-1.1829,  0.4065],
       [ 1.1055,  0.2856],
       [-1.6422, -0.8905]])

In [23]:
anotherFU = FeatureUnion([('original', SimpleTransformer(None, None, ['num_days', 'size_kw'])),
                      ('scaled',   make_pipeline(SimpleTransformer(None, None, ['num_days', 'size_kw']),
                                                 StandardScaler()))
                     ])

In [24]:
anotherFU.fit_transform(little)[:5]

array([[ 6.6400e+03,  5.8650e+00,  8.7910e-01, -7.6986e-02],
       [ 3.2250e+03,  4.9000e+00, -1.9219e+00, -3.7606e-01],
       [ 4.1260e+03,  7.4250e+00, -1.1829e+00,  4.0649e-01],
       [ 6.9160e+03,  7.0350e+00,  1.1055e+00,  2.8562e-01],
       [ 3.5660e+03,  3.2400e+00, -1.6422e+00, -8.9052e-01]])

### okay, have syntax for working featureunion that selects out

Now what do I want to do within GridSearchCV.

prep before the FeatureUnion:
   set y = df.cost_per_watt
   set X: get rid of 'cost_per_watt', 'scaleSize'; everything else is in
   

in the FeatureUnion:
```
arm 1
    select ['num_days', 'size_kw']
    scale
    polynomial(n)
arm 2
    select ['state_AZ', 'state_CA', 'state_CT', 'state_DE', 'state_FL', 'state_MA', 'state_MD',
            'state_MN', 'state_NH', 'state_NJ', 'state_NM', 'state_NV', 'state_NY',
            'state_OR', 'state_PA', 'state_TX', 'state_VT', 'state_WI']
    passthrough unmodified (0s and 1s)
```    




In [25]:
little.iloc[: , 2].shape

(1000,)

In [26]:
type(little)

pandas.core.frame.DataFrame

In [ ]:
theCols = ['num_days', 'size_kw', 'cost_per_watt', 'scaleSize', 'state_AZ',
       'state_CA', 'state_CT', 'state_DE', 'state_FL', 'state_MA', 'state_MD',
       'state_MN', 'state_NH', 'state_NJ', 'state_NM', 'state_NV', 'state_NY',
       'state_OR', 'state_PA', 'state_TX', 'state_VT', 'state_WI']

### Build a 30% sample.   But I'm not ready to use it.

In [27]:
dfModelAll.get_dtype_counts()

category    1
float64     3
int64       1
dtype: int64

In [28]:
dfMod = dfModelAll.sample(frac=0.3)

In [29]:
len(dfModelAll), len(dfMod)

(364212, 109264)

### recode categorical
  

In [30]:
 dfMod = pd.get_dummies(dfMod, drop_first=True); dfMod.head()

,num_days,size_kw,cost_per_watt,scaleSize,state_AZ,state_CA,state_CT,state_DE,state_FL,state_MA,state_MD,state_MN,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OR,state_PA,state_TX,state_VT,state_WI
row,,,,,,,,,,,,,,,,,,,,,,
282296,6639.0,9.35,3.750053,3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
145346,5474.0,2.40,5.000000,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
91441,4702.0,5.39,6.307978,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
299239,6696.0,4.68,3.845940,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
304533,6713.0,3.42,7.235380,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


#### Set up a sklearn pipeline

In [32]:
# 1000 row sample
little.head()

,num_days,size_kw,cost_per_watt,scaleSize,state_AZ,state_CA,state_CT,state_DE,state_FL,state_MA,state_MD,state_MN,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OR,state_PA,state_TX,state_VT,state_WI
row,,,,,,,,,,,,,,,,,,,,,,
282627,6640.0,5.865,4.728048,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22558,3225.0,4.900,8.500000,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
50856,4126.0,7.425,7.889918,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
360468,6916.0,7.035,4.718348,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
31636,3566.0,3.240,9.413580,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [33]:
# set up to model one feature (column vectors for both X and y)
# X = dfMod.values
# y = dfMod[['cost_per_watt']].values
# X.shape, y.shape

# for now work with little set
X = little.drop(['cost_per_watt', 'scaleSize'], axis='columns')
y = little[['cost_per_watt']]
X.shape, y.shape

((1000, 20), (1000, 1))

In [34]:
X[:3]

,num_days,size_kw,state_AZ,state_CA,state_CT,state_DE,state_FL,state_MA,state_MD,state_MN,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OR,state_PA,state_TX,state_VT,state_WI
row,,,,,,,,,,,,,,,,,,,,
282627,6640.0,5.865,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22558,3225.0,4.900,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
50856,4126.0,7.425,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [37]:
theStates = ['state_AZ', 'state_CA', 'state_CT', 'state_DE', 'state_FL', 'state_MA', 'state_MD',
            'state_MN', 'state_NH', 'state_NJ', 'state_NM', 'state_NV', 'state_NY',
            'state_OR', 'state_PA', 'state_TX', 'state_VT', 'state_WI']

In [ ]:
# # Setup the pipeline steps: steps
# steps = [('scaler', StandardScaler()),
#          ('poly', PolynomialFeatures()),
#          ('LR', LinearRegression())]

# # Create the pipeline: pipeline 
# pipeline = Pipeline(steps)

In [35]:
# take the data apart; numerical vars get scaled, then go to polynomial; state dummies pass through; 
# they get glued back together for the regression.
class Debug(BaseEstimator, TransformerMixin):

    def transform(self, X):
        print(pd.DataFrame(X).head(3))
        print(X.shape)
        return X

    def fit(self, X, y=None, **fit_params):
        return self

In [45]:
# now we go with FeatureUnion
# arm1 = Pipeline([('sel_nvars', SimpleTransformer(None, None, columns=['num_days', 'size_kw'])),
#                  ('dbg1a', Debug()),
#                  ('scaler', StandardScaler()),
#                  ('dbg1b', Debug()),
#                  ('poly', PolynomialFeatures()),
#                  ('dbg1c', Debug)
#                 ])
# arm2 = Pipeline([('sel_states', SimpleTransformer(None, None, columns=theStates)),
#                  ('dbg2', Debug())
#                 ])

theFU = FeatureUnion([('arm1',
                       Pipeline([('sel_nvars', SimpleTransformer(None, None, columns=['num_days', 'size_kw'])),
                                 ('dbg1a', Debug),
                                 ('scaler', StandardScaler()),
                                 
                                 #('poly', PolynomialFeatures()),
                                 ('dbg1c', Debug)])), 
                      ('arm2',
                       Pipeline([('sel_states', SimpleTransformer(None, None, columns=theStates)),
                                 ('dbg2', Debug())]) 
                      )])

# attach the regression to preprocessing
theBigPipe = Pipeline([
    ('sel_poly_join', theFU),
    ('dbg4', Debug()),
    ('LR', LinearRegression())
    ])

In [46]:
### test the feature union
theFU.fit_transform(little.head())

AttributeError: 'numpy.ndarray' object has no attribute 'fit'

In [ ]:
# take the data apart; numerical vars get scaled, then go to polynomial; state dummies pass through; 
# they get glued back together for the regression.

# # now we go with FeatureUnion
# arm1 = Pipeline([('simp_nvars', SimpleTransformer(None, None, columns=['num_days', 'size_kw'])),
#                  ('scaler', StandardScaler()),
#                  ('poly', PolynomialFeatures())
#                 ])
# arm2 = Pipeline([('simp_nvars', SimpleTransformer(None, None, columns=theStates))])

# theFU = FeatureUnion([('arm1', arm1), ('arm2', arm2)])

# # attach the regression to preprocessing
# theBigPipe = Pipeline([
#     ('sel_and_join', theFU),
#     ('LR', LinearRegression())
#     ])

In [39]:
type(theFU)

sklearn.pipeline.FeatureUnion

In [41]:
theBigPipe.get_params()

TypeError: get_params() missing 1 required positional argument: 'self'

In [ ]:
# what's in the pipeline?
theBigPipe.get_params()

In [ ]:
# what's in the pipeline?  poly degree is the default; gridsearchcv sets it...
theBigPipe.get_params()['sel_and_join__arm1__poly__degree']

In [ ]:
theBigPipe.get_params()['sel_and_join__arm1__poly__degree']

In [ ]:
# theBigPipe.get_params()

In [ ]:
# theBigPipe.set_params(sel_and_join__arm1__poly__degree=4)

In [ ]:
little.head()

In [ ]:
thing = theFU.fit_transform(little)

In [ ]:
type(thing)

In [ ]:
# Specify the hyperparameter space.
# Create the hyperparameter grid, just poly degree
poly_space = np.arange(1, 3)
# param_grid = {'poly__degree': poly_space}
param_grid = {'sel_and_join__arm1__poly__degree': poly_space}

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=None)

# Create the GridSearchCV object: gm_cv
gs_cv = GridSearchCV(theBigPipe, param_grid, cv=3, return_train_score=True)

# Fit to the training set
gs_cv.fit(X_train, y_train)

# Compute and print the metrics
r2 = gs_cv.score(X_test, y_test)
print("Best parameters: {}".format(gs_cv.best_params_))
print("test R squared: {}".format(r2))

In [ ]:
# this will show the results data structure
gs_cv.cv_results_

#### The pipeline above is working.  Next step is to add a scorer that produces multiple scores.  Example below.

#### instead of defining your own scorer to track multiple metrics, you can hand gridsearchcv a list of scoring fns to use (but you can only use the ones sklearn blesses).  the other way is to pass a dictionary...

In [ ]:
def RMSE(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Specify the hyperparameter space; .
# Create the hyperparameter grid, just poly degree
poly_space = np.arange(1, 11)
param_grid = {'poly__degree': poly_space}

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.40, random_state=None)

# Create the GridSearchCV object: gs_cv
scoring = {'Rsquared': 'r2',
           'RMSE': make_scorer(RMSE, greater_is_better=False) }

gs_cv = GridSearchCV(pipeline, param_grid, cv=3, 
                     scoring=scoring, return_train_score=True,
                     refit='Rsquared')

# Fit to the training set
gs_cv.fit(X_train, y_train)

# Compute and print the metrics
r2 = gs_cv.score(X_test, y_test)
print("Best parameters: {}".format(gs_cv.best_params_))
print("Best R squared: {}".format(r2))

# uncomment to see results ds
# gs_cv.cv_results_

In [ ]:
results = gs_cv.cv_results_

fig, ax = plt.subplots(figsize=(10, 10))
plt.title("GridSearchCV evaluating using multiple scorers simultaneously", fontsize=16)

plt.xlabel("polynomial degree")
plt.ylabel("Score")

# I think Seaborn ignores this...
plt.grid(True)

###            two__underscores
X_axis = np.array(results['param_poly__degree'].data, dtype=float)

for scorer, color in zip(sorted(scoring), ['g', 'r']):
    for sample, style in (('train', '.'), ('test', '-')):
        sample_score_mean = np.abs(results['mean_%s_%s' % (sample, scorer)])
        sample_score_std = results['std_%s_%s' % (sample, scorer)]
        ax.fill_between(X_axis, sample_score_mean - sample_score_std,
                        sample_score_mean + sample_score_std,
                        alpha=0.1 if sample == 'test' else 0, color=color)
        ax.plot(X_axis, sample_score_mean, style, color=color,
                alpha=1 if sample == 'test' else 0.7,
                label="%s (%s)" % (scorer, sample))

    best_index = np.abs(np.nonzero(results['rank_test_%s' % scorer] == 1)[0][0])
    best_score = np.abs(results['mean_test_%s' % scorer][best_index])

    # Plot a dotted vertical line at the best score for that scorer marked by x
    ax.plot([X_axis[best_index], ] * 2, [0, best_score],
            linestyle='-.', color=color, marker='x', markeredgewidth=3, ms=8)

    # Annotate the best score for that scorer
    ax.annotate("%0.2f" % best_score,
                (X_axis[best_index], best_score + 0.015))

plt.legend(loc="best")
plt.grid(False)
plt.show();

In [ ]:
list(zip(gs_cv.cv_results_['param_poly__degree'], 
         gs_cv.cv_results_['mean_test_Rsquared'], 
         np.abs(gs_cv.cv_results_['mean_test_RMSE'])))

In [ ]:
gs_cv.cv_results_

In [ ]:
poly_space = np.arange(6, 7)
param_grid = {'poly__degree': poly_space}

# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=None)

# Create the GridSearchCV object: gm_cv
gs_cv = GridSearchCV(pipeline, param_grid, cv=3, return_train_score=True)

# Fit to the training set
model = gs_cv.fit(X_train, y_train)

# Compute and print the metrics
r2 = gs_cv.score(X_test, y_test)
print("Best parameters: {}".format(gs_cv.best_params_))
print("Best R squared: {}".format(r2))


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.scatter(y_test, model.predict(X_test), marker='.', s=2, alpha=0.3);
ax1.plot(np.array([1,20]), np.array([1,20]), linewidth=1, color='red')

ax2.scatter(X_test[:, 0], y_test, marker='.', color='blue', s=2, alpha=0.3)
ax2.scatter(X_test[:, 0], model.predict(X_test), marker='.', color='red', s=2, alpha=0.3)
plt.show();